# MTM NBASIS ladder at fixed NXGRID 65 and fixed per-case WIDTH (Fusion_PhD-bhx7.10)

Question (user, 2026-10-02): for the best-capture arm (`extent_maxbasis_s100`: NXGRID 65, per-case WIDTH from the extent rule), does reducing NBASIS at the same NXGRID and WIDTH increase accuracy? Rungs `extent_maxbasis_s100_nb{12,20,30,40}` plus the existing NBASIS 59 (`extent_maxbasis_s100`); each rung differs only in `NBASIS_MAX = NBASIS_MIN`. Project definition: `results/mtm_maxbasis/config_nbladder.yaml`. NSTX_MTM `beta_q_shat_ky_n1000`.

**Population** (docs/mode_filtering.md): cases whose GS2 dominant mode `classify` labels MTM (T>0.15, omega<0, Fusion_PhD-lqwx); everything else is removed and counted. **Scoring**: GFTM's dominant mode (argmax gamma) whatever its type; capture = that mode is MTM by T + omega (`classify_T_omega`); a non-MTM dominant mode is a capture failure but its error still counts; no unstable mode is a MISS (error +inf). GS2-unstable means gamma > 1e-3.

## Imports and settings

In [ ]:
from pathlib import Path
import os
from dotenv import load_dotenv
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.ticker import StrMethodFormatter
from pyrokinetics import Pyro, PyroHypercube
from general_analysis import mode_classification as mc
from general_analysis.mode_classification import classify_T_omega

ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "pyproject.toml").is_file() and (p / "notebooks").is_dir())
plt.style.use(ROOT / "src/general_analysis/paper.mplstyle")
load_dotenv(ROOT / "local.env", override=True)

analysis_name = "mtm_nbladder"
data_root = Path(os.environ["GK_DATA_ROOT"]).expanduser()
run_template, project, case, draw = "Runs", "LATIN_HYPERCUBE", "NSTX_MTM", "beta_q_shat_ky_n1000"
reference = ("GS2", "pyro_cube_avg")
metadata_file = "pyroscan.json"
rungs = {12: "extent_maxbasis_s100_nb12", 20: "extent_maxbasis_s100_nb20", 30: "extent_maxbasis_s100_nb30",
         40: "extent_maxbasis_s100_nb40", 59: "extent_maxbasis_s100"}
gs2_unstable = 1e-3
n_boot, seed = 2000, 0

## Load data

GS2 reference and every rung through pyrokinetics.

In [ ]:
gs2_dir = data_root.joinpath(reference[0], run_template, project, case, draw, reference[1])
gs2_cube = PyroHypercube(pyroscan_json=gs2_dir / metadata_file, load_base_pyro=True)
gs2_cube.from_netcdf(gs2_dir / "cube.nc")
gs2 = gs2_cube.gk_output.data
data = {}
for nb, leaf in rungs.items():
    d = data_root.joinpath("GFTM", run_template, project, case, draw, leaf)
    cube = PyroHypercube(pyroscan_json=d / metadata_file, load_base_pyro=True)
    cube.load_gk_output(netcdf_file=d / "pyroscan.nc")
    data[nb] = cube.gk_output.data
    print(nb, dict(data[nb].sizes), {v: data[nb][v].dims for v in ("growth_rate", "mode_frequency", "tearing_parameter")})

## Classify the GS2 mode and fix the population

GS2's dominant mode is classified per case from its own run directory.

In [ ]:
names = [str(n) for n in gs2.sample_name.values]
gamma_gs2 = np.asarray(gs2.growth_rate.values, float).reshape(len(names), -1)[:, 0]
unstable = gamma_gs2 > gs2_unstable
gscan, gds = mc.load_gs2_cube(Path(gs2_cube.base_directory), "pyro_cube_avg")   # indicators from the cube, no run directory (Fusion_PhD-use8)
ind = mc.indicators(gscan, gds)
assert [str(n) for n in ind.sample_name.values] == names
gs2_class = np.where(unstable, mc.classify(ind), "").astype(object)
pop = unstable & (gs2_class == "MTM")
counts = {c: int((gs2_class[unstable] == c).sum()) for c in ("MTM", "KBM", "mixed")}
print(f"{len(names)} cases; GS2-unstable {unstable.sum()} ({len(names) - unstable.sum()} stable removed); classes among unstable {counts}")
print(f"MTM population n = {pop.sum()}; removed = {len(names) - pop.sum()}")

## Calculate: error, dominant-mode type and basis reach per rung

Dominant mode = argmax gamma over `mode`. T and omega are read for that mode; `classify_T_omega` (no per-mode flux weights for GFTM). Basis reach = WIDTH * sqrt(2 NBASIS - 1)... is context only; here the quadrature reach WIDTH*x_max(NXGRID) is also printed, read from each deck.

In [ ]:
def per_mode(ds, v):
    a = ds[v]
    if "time" in a.dims:
        a = a.isel(time=-1)
    return np.asarray(a.transpose("sample", "mode").values, float)

res = {}
for nb, ds in data.items():
    row = {str(n): i for i, n in enumerate(ds.sample_name.values)}
    take = np.array([row.get(n, -1) for n in names])
    ok = take >= 0
    t = np.maximum(take, 0)
    g = per_mode(ds, "growth_rate")
    gm = np.where(np.isfinite(g), g, -np.inf)
    k = gm.argmax(axis=1)                        # dominant mode, never isel(mode=0)
    idx = np.arange(len(k))
    gdom = np.where(ok, gm.max(axis=1)[t], np.nan)
    T = per_mode(ds, "tearing_parameter")[idx, k][t]
    om = per_mode(ds, "mode_frequency")[idx, k][t]
    is_mtm = np.array([gd > 0 and classify_T_omega(dict(T=a, omega=b)) == "MTM" for gd, a, b in zip(gdom, T, om)]) & ok
    with np.errstate(divide="ignore", invalid="ignore"):
        ae = np.where(gdom > 0, np.abs(gdom - gamma_gs2), np.inf)          # miss = +inf
        lg = np.where(gdom > 0, np.abs(np.log10(gdom / gamma_gs2)), np.inf)
    leaf_dir = data_root.joinpath("GFTM", run_template, project, case, draw, rungs[nb])
    decks = [Pyro(gk_file=leaf_dir / n / "input.gftm").gk_input.data for n in names]
    nx = np.array([int(d["nxgrid"]) for d in decks]); w = np.array([float(d["width"]) for d in decks])
    nbd = np.array([int(d["nbasis_max"]) for d in decks])
    assert (nx == 65).all() and (nbd == nb).all()
    res[nb] = dict(ae=ae, lg=lg, mtm=is_mtm, width=w, reach=w * np.polynomial.hermite.hermgauss(129)[0].max() / np.pi,
                   brs=w * np.sqrt(2 * nb - 1) / np.pi, ok=ok)
    print(nb, "decks", int(ok.sum()), "| misses in pop", int(np.isinf(ae[pop]).sum()), "| dominant not MTM in pop", int((~is_mtm[pop]).sum()))
w0 = res[59]["width"]
print("WIDTH identical across rungs:", all(np.allclose(r["width"], w0) for r in res.values()))

## Score with bootstrap intervals

Same resampled cases for every rung (paired). medlog = median |log10 gamma ratio|; LogMed = log10 of the median absolute error |gamma_GFTM - gamma_GS2|; misses are +inf and kept.

In [ ]:
rng = np.random.default_rng(seed)
b = rng.integers(0, pop.sum(), (n_boot, pop.sum()))
sc = {}
for nb, r in res.items():
    ae, lg, cap = r["ae"][pop], r["lg"][pop], r["mtm"][pop]
    with np.errstate(divide="ignore"):
        sc[nb] = dict(medlog=np.median(lg), medlog_ci=np.percentile(np.median(lg[b], 1), [2.5, 97.5]),
                      logmed=np.log10(np.median(ae)), logmed_ci=np.log10(np.percentile(np.median(ae[b], 1), [2.5, 97.5])),
                      capture=cap.mean(), cap_ci=np.percentile(cap[b].mean(1), [2.5, 97.5]), misses=int(np.isinf(ae).sum()))
print(f"MTM population n={pop.sum()}")
print("NBASIS | medlog [CI] | LogMed [CI] | capture [CI] | misses | median reach/pi | median WIDTH*sqrt(2N-1)/pi")
for nb, s in sc.items():
    r = res[nb]
    print(f"{nb:3d} | {s['medlog']:.3f} [{s['medlog_ci'][0]:.3f},{s['medlog_ci'][1]:.3f}] | {s['logmed']:.3f} [{s['logmed_ci'][0]:.3f},{s['logmed_ci'][1]:.3f}] | "
          f"{s['capture']:.3f} [{s['cap_ci'][0]:.3f},{s['cap_ci'][1]:.3f}] | {s['misses']} | {np.median(r['reach'][pop]):.2f} | {np.median(r['brs'][pop]):.2f}")
print("paired medlog difference vs 59 [CI]:")
for nb in rungs:
    d = [np.median(res[nb]["lg"][pop][b], 1) - np.median(res[59]["lg"][pop][b], 1)]
    print(nb, f"{sc[nb]['medlog'] - sc[59]['medlog']:+.3f}", np.round(np.percentile(d[0], [2.5, 97.5]), 3))

## Plot

Figure 1: LogMed |gamma error| against NBASIS. Figure 2: medlog against NBASIS. Figure 3: capture against NBASIS. One line each, GS2-MTM population, n1000.

In [ ]:
x = np.array(list(rungs))
def line(key, ci, ylabel, fname):
    fig, ax = plt.subplots(figsize=(6, 4.5))
    y = np.array([sc[n][key] for n in x]); e = np.array([sc[n][ci] for n in x])
    ax.errorbar(x, y, yerr=[y - e[:, 0], e[:, 1] - y], marker="o", capsize=3)
    ax.set_xscale("log"); ax.set_xticks(x); ax.xaxis.set_major_formatter(StrMethodFormatter("{x:g}")); ax.minorticks_off()
    ax.set_xlabel("NBASIS", fontsize=16); ax.set_ylabel(ylabel, fontsize=16); ax.set_title("NSTX_MTM", fontsize=16)
    ax.tick_params(labelsize=14)
    return fig
fig1 = line("logmed", "logmed_ci", r"LogMed $|\gamma_{\rm GFTM}-\gamma_{\rm GS2}|$", "")
fig2 = line("medlog", "medlog_ci", r"medlog $|\log_{10}\gamma$ ratio$|$", "")
fig3 = line("capture", "cap_ci", "MTM capture", "")
plt.show()

## Save

In [ ]:
output_dir = ROOT / "Plots" / analysis_name
output_dir.mkdir(parents=True, exist_ok=True)
for f, n in ((fig1, "fig1_logmed_vs_nbasis"), (fig2, "fig2_medlog_vs_nbasis"), (fig3, "fig3_capture_vs_nbasis")):
    f.tight_layout(); f.savefig(output_dir / f"{n}.png", dpi=150)

## Interpretation

On the 711 GS2-MTM cases (289 removed: 96 GS2-stable, 136 KBM, 57 mixed), fewer basis functions at fixed NXGRID 65 and fixed per-case WIDTH make every score worse, not better. medlog 0.639 / 0.453 / 0.405 / 0.390 / 0.341 for NBASIS 12 / 20 / 30 / 40 / 59, paired differences to 59 all positive with CIs excluding 0; capture falls 0.824 to 0.408; misses (no unstable mode) rise from 1 to 78-115. LogMed |gamma error| is flatter (-0.79 to -0.91) and not monotone between 12 and 40, because it only sees the median of the absolute error; NBASIS 59 is still the lowest. nb40 has one case (iteration_307) that never ran, scored as a miss.